# Q5: Self-Consistency Decoding for Math Reasoning

Instead of greedy decoding one answer, we sample N reasoning paths and majority-vote the final answer.

No extra training needed — this is purely an inference-time improvement.

In [1]:
!nvidia-smi

Thu Mar 12 15:49:31 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   29C    P0             46W /  600W |       0MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os
os.chdir('/content/drive/MyDrive/nlu_assignment_cot')

In [4]:
!pip install -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 153.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.9/511.9 kB 51.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 504.9/504.9 kB 57.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.1/209.1 kB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 73.8 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.5.0
    Uninstalling huggingface_hub-1.5.0:
      Successfully uninstalled huggingface_hub-1.5.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
  Attempting uninstall: peft
    Found existing installation: peft 0.18.1
    Uninstalling peft-0.18.1:
      Successfully uninstalled peft-0.18.1


## Instruct SFT+GRPO  SC@10

In [6]:
!python evaluation/main_sc.py \
    --model_signature Qwen/Qwen2.5-0.5B-Instruct \
    --sft_adapter_path ./checkpoints/Qwen/Qwen2.5-0.5B-Instruct-sft \
    --grpo_adapter_path ./checkpoints/Qwen/Qwen2.5-0.5B-Instruct-sft_grpo \
    --output_path ./outputs/Qwen/Qwen2.5-0.5B-Instruct-sft_grpo-sc5 \
    --num_samples 5 \
    --temperature 0.5

2026-03-12 15:52:52.727094: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-12 15:52:52.735470: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773330772.744927    3121 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773330772.748098    3121 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773330772.756184    3121 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [7]:
!python evaluation/main_sc.py \
    --model_signature Qwen/Qwen2.5-0.5B-Instruct \
    --sft_adapter_path ./checkpoints/Qwen/Qwen2.5-0.5B-Instruct-sft \
    --grpo_adapter_path ./checkpoints/Qwen/Qwen2.5-0.5B-Instruct-sft_grpo \
    --output_path ./outputs/Qwen/Qwen2.5-0.5B-Instruct-sft_grpo-sc5-t7 \
    --num_samples 5 \
    --temperature 0.7

2026-03-12 15:56:36.338258: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-12 15:56:36.346900: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773330996.356576    4385 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773330996.359826    4385 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773330996.368037    4385 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [8]:
!python evaluation/main_sc.py \
    --model_signature Qwen/Qwen2.5-0.5B-Instruct \
    --sft_adapter_path ./checkpoints/Qwen/Qwen2.5-0.5B-Instruct-sft \
    --grpo_adapter_path ./checkpoints/Qwen/Qwen2.5-0.5B-Instruct-sft_grpo \
    --output_path ./outputs/Qwen/Qwen2.5-0.5B-Instruct-sft_grpo-sc10-t5 \
    --num_samples 10 \
    --temperature 0.5

2026-03-12 16:01:17.799090: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-12 16:01:17.807728: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773331277.817365    5833 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773331277.820609    5833 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773331277.828889    5833 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [9]:
!python evaluation/main_sc.py \
    --model_signature Qwen/Qwen2.5-0.5B-Instruct \
    --sft_adapter_path ./checkpoints/Qwen/Qwen2.5-0.5B-Instruct-sft \
    --grpo_adapter_path ./checkpoints/Qwen/Qwen2.5-0.5B-Instruct-sft_grpo \
    --output_path ./outputs/Qwen/Qwen2.5-0.5B-Instruct-sft_grpo-sc10-t7 \
    --num_samples 10 \
    --temperature 0.7

2026-03-12 16:05:09.590163: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-12 16:05:09.598824: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773331509.608483    7083 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773331509.611747    7083 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773331509.620008    7083 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [13]:
!python evaluation/main_sc.py \
    --model_signature Qwen/Qwen2.5-0.5B-Instruct \
    --sft_adapter_path ./checkpoints/Qwen/Qwen2.5-0.5B-Instruct-sft \
    --grpo_adapter_path ./checkpoints/Qwen/Qwen2.5-0.5B-Instruct-sft_grpo \
    --output_path ./outputs/Qwen/Qwen2.5-0.5B-Instruct-sft_grpo-sc10-t3 \
    --num_samples 10 \
    --temperature 0.3

2026-03-12 16:22:02.399207: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-12 16:22:02.407893: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773332522.417506   12397 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773332522.420750   12397 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773332522.429083   12397 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [10]:
!python evaluation/main_sc.py \
    --model_signature Qwen/Qwen2.5-0.5B-Instruct \
    --sft_adapter_path ./checkpoints/Qwen/Qwen2.5-0.5B-Instruct-sft \
    --grpo_adapter_path ./checkpoints/Qwen/Qwen2.5-0.5B-Instruct-sft_grpo \
    --output_path ./outputs/Qwen/Qwen2.5-0.5B-Instruct-sft_grpo-sc20-t5 \
    --num_samples 20 \
    --temperature 0.5

2026-03-12 16:09:01.593903: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-12 16:09:01.602918: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773331741.612662    8327 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773331741.615948    8327 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773331741.624224    8327 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [11]:
!python evaluation/main_sc.py \
    --model_signature Qwen/Qwen2.5-0.5B-Instruct \
    --sft_adapter_path ./checkpoints/Qwen/Qwen2.5-0.5B-Instruct-sft \
    --grpo_adapter_path ./checkpoints/Qwen/Qwen2.5-0.5B-Instruct-sft_grpo \
    --output_path ./outputs/Qwen/Qwen2.5-0.5B-Instruct-sft_grpo-sc20-t7 \
    --num_samples 20 \
    --temperature 0.7

2026-03-12 16:13:22.960156: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-12 16:13:22.968702: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773332002.978235    9695 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773332002.981456    9695 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773332002.989672    9695 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [12]:
!python evaluation/main_sc.py \
    --model_signature Qwen/Qwen2.5-0.5B-Instruct \
    --sft_adapter_path ./checkpoints/Qwen/Qwen2.5-0.5B-Instruct-sft \
    --grpo_adapter_path ./checkpoints/Qwen/Qwen2.5-0.5B-Instruct-sft_grpo \
    --output_path ./outputs/Qwen/Qwen2.5-0.5B-Instruct-sft_grpo-sc20-t3 \
    --num_samples 20 \
    --temperature 0.3

2026-03-12 16:18:12.468147: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-12 16:18:12.476748: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773332292.486304   11192 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773332292.489516   11192 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773332292.497723   11192 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [14]:
!python evaluation/main_sc.py \
    --model_signature Qwen/Qwen2.5-0.5B-Instruct \
    --sft_adapter_path ./checkpoints/Qwen/Qwen2.5-0.5B-Instruct-sft \
    --grpo_adapter_path ./checkpoints/Qwen/Qwen2.5-0.5B-Instruct-sft_grpo \
    --output_path ./outputs/Qwen/Qwen2.5-0.5B-Instruct-sft_grpo-sc40-t3 \
    --num_samples 40 \
    --temperature 0.5

2026-03-12 16:25:07.195083: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-12 16:25:07.203756: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773332707.213360   13410 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773332707.216609   13410 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773332707.224894   13410 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

## Base SFT+GRPO  SC@10

In [ ]:
!python evaluation/main_sc.py \
    --model_signature Qwen/Qwen2.5-0.5B \
    --sft_adapter_path ./checkpoints/Qwen/Qwen2.5-0.5B-base-sft \
    --grpo_adapter_path ./checkpoints/Qwen/Qwen2.5-0.5B-base-sft_grpo \
    --output_path ./outputs/Qwen/Qwen2.5-0.5B-base-sft_grpo-sc5 \
    --num_samples 5 \
    --temperature 0.5

## Collect & Compare Results

In [15]:
import os, re

rows = [
    ("Instruct SFT+GRPO (greedy)",       "outputs/Qwen/Qwen2.5-0.5B-Instruct-sft_grpo"),
    ("Instruct SFT+GRPO (SC@5, T=0.5)",  "outputs/Qwen/Qwen2.5-0.5B-Instruct-sft_grpo-sc5"),
    ("Instruct SFT+GRPO (SC@5, T=0.7)",  "outputs/Qwen/Qwen2.5-0.5B-Instruct-sft_grpo-sc5-t7"),
    ("Instruct SFT+GRPO (SC@10, T=0.5)", "outputs/Qwen/Qwen2.5-0.5B-Instruct-sft_grpo-sc10-t5"),
    ("Instruct SFT+GRPO (SC@10, T=0.7)", "outputs/Qwen/Qwen2.5-0.5B-Instruct-sft_grpo-sc10-t7"),
    ("Instruct SFT+GRPO (SC@10, T=0.3)", "outputs/Qwen/Qwen2.5-0.5B-Instruct-sft_grpo-sc10-t3"),
    ("Instruct SFT+GRPO (SC@20, T=0.5)", "outputs/Qwen/Qwen2.5-0.5B-Instruct-sft_grpo-sc20-t5"),
    ("Instruct SFT+GRPO (SC@20, T=0.7)", "outputs/Qwen/Qwen2.5-0.5B-Instruct-sft_grpo-sc20-t7"),
    ("Instruct SFT+GRPO (SC@20, T=0.3)", "outputs/Qwen/Qwen2.5-0.5B-Instruct-sft_grpo-sc20-t3"),
    ("Instruct SFT+GRPO (SC@40, T=0.3)", "outputs/Qwen/Qwen2.5-0.5B-Instruct-sft_grpo-sc40-t3"),
]

print(f"{'Model':<38} {'Accuracy':>10}")
print('=' * 50)
for label, path in rows:
    txt = os.path.join(path, 'results.txt')
    if not os.path.exists(txt):
        print(f"{label:<38} {'(not run)':>10}")
        continue
    with open(txt) as f:
        content = f.read()
    m = re.search(r'Overall Accuracy.*?([\d.]+%)', content)
    print(f"{label:<38} {m.group(1) if m else 'N/A':>10}")

Model                                    Accuracy
Instruct SFT+GRPO (greedy)                 32.00%
Instruct SFT+GRPO (SC@5, T=0.5)            37.00%
Instruct SFT+GRPO (SC@5, T=0.7)            34.00%
Instruct SFT+GRPO (SC@10, T=0.5)           43.00%
Instruct SFT+GRPO (SC@10, T=0.7)           42.00%
Instruct SFT+GRPO (SC@10, T=0.3)           39.00%
Instruct SFT+GRPO (SC@20, T=0.5)           48.00%
Instruct SFT+GRPO (SC@20, T=0.7)           46.00%
Instruct SFT+GRPO (SC@20, T=0.3)           35.00%
Instruct SFT+GRPO (SC@40, T=0.3)           45.00%
